# Fake news detection

## A. Upload package and load dataset

### A.1 Packages 

In [2]:
import pandas as pd


### A.2 Dataset

In [3]:

# Colonnes du dataset  récupéré depuis le readme.md du dataset
COLONNES = [
    "id",                   
    "label",                
    "statement",            
    "subject",              
    "speaker",              
    "job_title",            
    "state",                
    "party",                
    "barely_true_counts",   
    "false_counts",         
    "half_true_counts",     
    "mostly_true_counts",   
    "pants_on_fire_counts",
    "context"               
]

# Chargement des données 
train = pd.read_csv("data/train.tsv", sep="\t", header=None, names=COLONNES)
valid = pd.read_csv("data/valid.tsv", sep="\t", header=None, names=COLONNES)
test  = pd.read_csv("data/test.tsv",  sep="\t", header=None, names=COLONNES)

# Vérifiation
print("Chargement réussi !")
print(f"   Train : {train.shape[0]} lignes, {train.shape[1]} colonnes")
print(f"   Valid : {valid.shape[0]} lignes, {valid.shape[1]} colonnes")
print(f"   Test  : {test.shape[0]} lignes, {test.shape[1]} colonnes")

Chargement réussi !
   Train : 10240 lignes, 14 colonnes
   Valid : 1284 lignes, 14 colonnes
   Test  : 1267 lignes, 14 colonnes


VÉRIFICATION DES COLONNES

In [ ]:
print("Colonnes Train :", train.columns.tolist())
print("Colonnes Valid :", valid.columns.tolist())
print("Colonnes Test :", test.columns.tolist())

# Vérification automatiqu
if train.columns.tolist() == valid.columns.tolist() == test.columns.tolist():
    print("\n Les 3 datasets ont exactement les mêmes colonnes !")
else:
    print("\n Attention, les colonnes sont différentes !")

Colonnes Train : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Valid : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Test : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']

 Les 3 datasets ont exactement les mêmes colonnes !


Apercu des données

In [9]:
print("Apercu des 5 premières lignes du dataset train :")
train.head(5)

Apercu des 5 premières lignes du dataset train :


,id,label,statement,subject,speaker,job_title,state,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


 VALEURS MANQUANTES SUR LES 3 DATASETS




In [25]:
import pandas as pd
from IPython.display import display

for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    print(f"\n### {nom}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    result = pd.DataFrame({
        "Valeurs manquantes": missing,
        "Pourcentage (%)": missing_pct
    })
    
    display(result[result["Valeurs manquantes"] > 0])


### Train


,Valeurs manquantes,Pourcentage (%)
job_title,2898,28.30
state,2210,21.58
barely_true_counts,2,0.02
false_counts,2,0.02
half_true_counts,2,0.02
mostly_true_counts,2,0.02
pants_on_fire_counts,2,0.02



### Valid


,Valeurs manquantes,Pourcentage (%)
job_title,345,26.87
state,279,21.73



### Test


,Valeurs manquantes,Pourcentage (%)
job_title,325,25.65
state,262,20.68


Après analyse des 3 datasets (train, valid, test), on identifie les valeurs manquantes suivantes

**`job_title` et `state` (≈25%) → on ne fait rien**
Ces colonnes ne seront pas utilisées dans notre pipeline de classification.
Le texte principal (`statement`) suffit pour entraîner le modèle.
Supprimer ces lignes ferait perdre 25% des données inutilement.

**`context`, `speaker`, `party`, `subject` (< 1%) → on remplace par une valeur neutre**
Ces colonnes sont utiles notamment pour l'analyse des biais (par parti, par speaker).
On remplace les manquants par `"unknown"` ou `""` pour ne perdre aucune ligne.
La règle d'or : on ne supprime une ligne que si elle est complètement inutilisable.

**`statement` et `label` → aucune action nécessaire**
Ces deux colonnes sont les plus importantes du projet :
`statement` est le texte qu'on classifie, `label` est ce qu'on cherche à prédire.
Heureusement elles ne contiennent aucune valeur manquante sur les 3 datasets.

Nettoyage des données

In [17]:
for df in [train, valid, test]:
    df["context"] = df["context"].fillna("")
    df["speaker"] = df["speaker"].fillna("unknown")
    df["party"]   = df["party"].fillna("unknown")
    df["subject"] = df["subject"].fillna("")

print(" Nettoyage terminé ")
print(f"   Train : {len(train)} lignes")
print(f"   Valid : {len(valid)} lignes")
print(f"   Test  : {len(test)} lignes")

# Vérification : plus aucune valeur manquante sur les colonnes utiles
cols_utiles = ["label", "statement", "speaker", "party", "context", "subject"]
print("\n### Vérification finale sur les colonnes utiles")
for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    nb = df[cols_utiles].isnull().sum().sum()
    print(f"   {nom} : {nb} valeurs manquantes restantes")

 Nettoyage terminé 
   Train : 10240 lignes
   Valid : 1284 lignes
   Test  : 1267 lignes

### Vérification finale sur les colonnes utiles
   Train : 0 valeurs manquantes restantes
   Valid : 0 valeurs manquantes restantes
   Test : 0 valeurs manquantes restantes


## Distribution des labels

On analyse ici la répartition des 6 labels de véracité dans le train set.
C'est une étape cruciale pour détecter un éventuel déséquilibre des classes
qui pourrait biaiser notre modèle.

In [ ]:

import plotly.express as px

ORDRE_LABELS = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

COULEURS = [
    "#d73027", 
    "#f46d43",  
    "#fdae61", 
    "#fee08b",  
    "#a6d96a", 
    "#1a9850"   
]

counts = train["label"].value_counts().reindex(ORDRE_LABELS)

fig = px.bar(
    x=counts.index,
    y=counts.values,
    color=counts.index,
    color_discrete_sequence=COULEURS,
    title="Distribution des labels — Train set",
    labels={"x": "Label de véracité", "y": "Nombre de déclarations"},
    text=counts.values
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

# Résumé chiffré
print("\n### Résumé")
for label, count in counts.items():
    pct = count / len(train) * 100
    print(f"   {label:<20} : {count} ({pct:.1f}%)")


### Résumé
   pants-fire           : 839 (8.2%)
   false                : 1995 (19.5%)
   barely-true          : 1654 (16.2%)
   half-true            : 2114 (20.6%)
   mostly-true          : 1962 (19.2%)
   true                 : 1676 (16.4%)


**Bonne nouvelle** : la majorité des classes sont relativement équilibrées
entre 16% et 20%, ce qui est favorable pour l'entraînement du modèle.

**Point d'attention** : `pants-fire` est sous-représenté avec seulement 8.2%.
C'est deux fois moins que les autres classes. Le modèle aura donc
moins d'exemples pour apprendre à détecter les mensonges flagrants.

On devra gérer ce léger déséquilibre avec l'une de ces stratégies :
- `class_weight="balanced"` dans scikit-learn
- Regroupement des labels en binaire (fake / true)
- Oversampling de la classe minoritaire


A voir après 

##  Analyse des speakers et des partis politiques

On analyse ici qui parle le plus dans le dataset et
quel est le lien entre le parti politique et la véracité des déclarations.
C'est aussi la base de notre analyse des biais plus tard.

In [19]:
top_speakers = train["speaker"].value_counts().head(10).reset_index()
top_speakers.columns = ["speaker", "count"]

fig = px.bar(
    top_speakers,
    x="count",
    y="speaker",
    orientation="h",
    title="Top 10 speakers les plus présents",
    labels={"count": "Nombre de déclarations", "speaker": "Speaker"},
    color="count",
    color_continuous_scale="Blues",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.show()

Barack Obama est de loin le speaker le plus représenté avec 488 déclarations,
suivi de Donald Trump (273) et Hillary Clinton (239).
On note aussi "chain-email" qui représente des emails viraux anonymes.

Biais potentiel : le modèle risque de sur-apprendre
les patterns propres à Obama vu sa surreprésentation dans le dataset.

In [20]:

top_partis = train["party"].value_counts().head(8).reset_index()
top_partis.columns = ["party", "count"]

fig = px.bar(
    top_partis,
    x="party",
    y="count",
    title="Distribution des déclarations par parti politique",
    labels={"count": "Nombre de déclarations", "party": "Parti"},
    color="count",
    color_continuous_scale="Purples",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Le dataset est dominé par deux partis :
- Republican : 4497 déclarations (44%)
- Democrat : 3336 déclarations (33%)
- None : 1744 déclarations (17%)

Les autres partis (independent, libertarian, activist) sont très minoritaires.

Biais potentiel : le modèle sera beaucoup mieux calibré
pour les déclarations républicaines et démocrates que pour les autres.

In [21]:

# Garder uniquement les 5 partis les plus représentés
top_5_partis = train["party"].value_counts().head(5).index

df_partis = train[train["party"].isin(top_5_partis)]

# Calculer la proportion de chaque label par parti
df_grouped = (
    df_partis
    .groupby(["party", "label"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    df_grouped,
    x="party",
    y="count",
    color="label",
    title="Véracité des déclarations par parti politique (top 5)",
    labels={"count": "Nombre de déclarations", "party": "Parti", "label": "Label"},
    category_orders={"label": ORDRE_LABELS},
    color_discrete_sequence=COULEURS,
    barmode="group"
)

fig.show()

On observe une différence notable entre les partis :
- Les républicains ont plus de `pants-fire` et `false` que les démocrates
- Les démocrates ont proportionnellement plus de `mostly-true` 

Attention : cela ne veut pas dire qu'un parti ment plus que l'autre.
Cela peut refléter un biais dans les sources annotées par PolitiFact,
qui a peut-être vérifié davantage de déclarations républicaines controversées.

## Analyse de la longueur des déclarations

On analyse ici la longueur des déclarations en nombre de mots.
C'est important car les textes courts posent un défi particulier
pour les modèles NLP qui ont besoin de contexte pour bien classifier.

In [ ]:

# Calcul du nombre de mots
train["nb_mots"] = train["statement"].str.split().str.len()

print("### Statistiques de longueur")
print(train["nb_mots"].describe().round(2))

# Histogramme
fig = px.histogram(
    train,
    x="nb_mots",
    nbins=50,
    title="Distribution de la longueur des déclarations",
    labels={"nb_mots": "Nombre de mots", "count": "Fréquence"},
    color_discrete_sequence=["#4C72B0"]
)

fig.add_vline(
    x=train["nb_mots"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text=f"Moyenne : {train['nb_mots'].mean():.1f} mots",
    annotation_position="top right"
)

fig.show()

### Statistiques de longueur
count    10240.00
mean        18.01
std          9.66
min          2.00
25%         12.00
50%         17.00
75%         22.00
max        467.00
Name: nb_mots, dtype: float64


es déclarations sont en moyenne très courtes : seulement 18 mots.
75% des déclarations font moins de 22 mots ce qui confirme
que le dataset est composé majoritairement de phrases courtes.

Défi pour la modélisation : les textes courts contiennent
peu de contexte, ce qui rend la classification plus difficile.
C'est une des raisons pour lesquelles BERT sera plus performant
que TF-IDF — il capte mieux le sens avec peu de mots.

Valeur maximale de 467 mots : il existe quelques outliers
très longs qui pourraient perturber certains modèles.
On les gardera mais on en tiendra compte dans l'analyse des erreurs.

In [28]:

moyenne_par_label = (
    train.groupby("label")["nb_mots"]
    .mean()
    .reindex(ORDRE_LABELS)
    .round(1)
    .reset_index()
)
moyenne_par_label.columns = ["label", "moyenne_mots"]

fig = px.bar(
    moyenne_par_label,
    x="label",
    y="moyenne_mots",
    color="label",
    title="Longueur moyenne des déclarations par label",
    labels={"moyenne_mots": "Nombre de mots moyen", "label": "Label"},
    color_discrete_sequence=COULEURS,
    text="moyenne_mots"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Les différences de longueur entre les labels sont très faibles
(entre 17 et 18.8 mots en moyenne).

On ne peut donc pas utiliser la longueur comme feature
discriminante pour détecter les fake news.

Cela confirme que le modèle devra s'appuyer uniquement
sur le contenu sémantique des mots et non sur la longueur.